## Code chính tải dataset (chưa lọc và tiền xử lý thêm) 
Bao gồm chia theo Nhóm bệnh -> Mã bệnh nhân -> Ngày khám/chụp
Tải song song với 32 worker, có thể tăng nếu muốn

In [2]:
import pandas as pd
import requests
import threading
import time

from pathlib import Path
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

# =====================================================
# CONFIG
# =====================================================

EXCEL_FILE = r"C:\Users\lebat\Documents\Github\xbone-net\data\CTCH\Data_CTCH.xlsx"

OUTPUT_ROOT = Path(r"C:\Users\lebat\Documents\Github\xbone-net\data\CTCH\data_grouped")

MAX_WORKERS = 32
MAX_RETRIES = 8

OUTPUT_ROOT.mkdir(exist_ok=True)

# =====================================================
# READ EXCEL
# =====================================================

image_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Image Links"
)

patient_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Patients",
    header=1
)

image_df.columns = image_df.columns.astype(str).str.strip()
patient_df.columns = patient_df.columns.astype(str).str.strip()

# =====================================================
# PATIENT MAP
# =====================================================

patient_map = {}

for _, row in patient_df.iterrows():

    if pd.isna(row["Mã hồ sơ"]):
        continue

    ma_ho_so = str(row["Mã hồ sơ"]).strip()

    nhom_benh = row.get("Nhóm bệnh")

    if pd.isna(nhom_benh):
        nhom_benh = "Unknown"

    patient_map[ma_ho_so] = str(nhom_benh).strip()

# =====================================================
# LOG FILES
# =====================================================

duplicate_log = OUTPUT_ROOT / "duplicate_files.txt"
failed_log = OUTPUT_ROOT / "failed_urls.txt"

duplicate_log.write_text("", encoding="utf-8")
failed_log.write_text("", encoding="utf-8")

filename_lock = Lock()
log_lock = Lock()

# =====================================================
# SESSION POOL
# =====================================================

thread_local = threading.local()


def get_session():

    if not hasattr(thread_local, "session"):

        session = requests.Session()

        adapter = requests.adapters.HTTPAdapter(
            pool_connections=100,
            pool_maxsize=100
        )

        session.mount("http://", adapter)
        session.mount("https://", adapter)

        thread_local.session = session

    return thread_local.session


# =====================================================
# HELPERS
# =====================================================

def clean_name(text):

    text = str(text).strip()

    invalid_chars = '<>:"/\\|?*'

    for c in invalid_chars:
        text = text.replace(c, "_")

    return text


def format_exam_date(value):

    if pd.isna(value):
        return "UnknownDate"

    try:
        dt = pd.to_datetime(value)
        return dt.strftime("%d%m%Y")

    except:
        value = str(value)

        return (
            value.replace("/", "")
                 .replace("-", "")
                 .replace(".", "")
                 .replace(" ", "")
        )


def get_filename(url):

    filename = unquote(
        Path(
            urlparse(url).path
        ).name
    )

    return filename


def get_unique_filepath(filepath):

    with filename_lock:

        if not filepath.exists():
            return filepath, False

        stem = filepath.stem
        suffix = filepath.suffix

        counter = 1

        while True:

            candidate = (
                filepath.parent /
                f"{stem}_{counter}{suffix}"
            )

            if not candidate.exists():
                return candidate, True

            counter += 1


# =====================================================
# DOWNLOAD WITH RETRY
# =====================================================

def download_file(url, filepath):

    session = get_session()

    last_error = None

    for attempt in range(MAX_RETRIES):

        try:

            response = session.get(
                url,
                timeout=(10, 180),
                stream=True
            )

            response.raise_for_status()

            with open(filepath, "wb") as f:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:
                        f.write(chunk)

            return True, None

        except Exception as e:

            last_error = e

            wait_time = min(
                2 ** attempt,
                60
            )

            time.sleep(wait_time)

    return False, str(last_error)


# =====================================================
# PROCESS ROW
# =====================================================

def process_row(idx, row):

    try:

        ma_ho_so = str(
            row["Mã hồ sơ"]
        ).strip()

        ngay_kham = format_exam_date(
            row["Ngày khám"]
        )

        url = row["Link tải ảnh"]

        if pd.isna(url):
            return "missing"

        nhom_benh = patient_map.get(
            ma_ho_so,
            "Unknown"
        )

        nhom_benh = clean_name(
            nhom_benh
        )

        target_dir = (
            OUTPUT_ROOT
            / nhom_benh
            / clean_name(ma_ho_so)
            / ngay_kham
        )

        target_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        filename = get_filename(url)

        if not filename:
            filename = f"image_{idx}.jpg"

        filepath = target_dir / filename

        filepath, duplicated = (
            get_unique_filepath(filepath)
        )

        if duplicated:

            with log_lock:

                with open(
                    duplicate_log,
                    "a",
                    encoding="utf-8"
                ) as f:

                    f.write(
                        f"{filename} -> {filepath}\n"
                    )

        # resume
        if (
            filepath.exists()
            and filepath.stat().st_size > 0
        ):
            return "skip"

        success, error = download_file(
            url,
            filepath
        )

        if not success:

            with log_lock:

                with open(
                    failed_log,
                    "a",
                    encoding="utf-8"
                ) as f:

                    f.write(
                        f"{idx}\t{url}\n"
                    )

            return "failed"

        return "success"

    except Exception:

        with log_lock:

            with open(
                failed_log,
                "a",
                encoding="utf-8"
            ) as f:

                f.write(
                    f"{idx}\t{url}\n"
                )

        return "failed"


# =====================================================
# RUN
# =====================================================

total = len(image_df)

success_count = 0
failed_count = 0
skip_count = 0

print(f"Total images: {total}")
print(f"Workers: {MAX_WORKERS}")

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = [

        executor.submit(
            process_row,
            idx,
            row
        )

        for idx, row in image_df.iterrows()
    ]

    completed = 0

    for future in as_completed(futures):

        completed += 1

        result = future.result()

        if result == "success":
            success_count += 1

        elif result == "failed":
            failed_count += 1

        elif result == "skip":
            skip_count += 1

        if completed % 100 == 0:

            print(
                f"[{completed}/{total}] "
                f"success={success_count} "
                f"failed={failed_count} "
                f"skip={skip_count}"
            )

print("\n==========================")
print("DOWNLOAD FINISHED")
print("==========================")
print(f"Total   : {total}")
print(f"Success : {success_count}")
print(f"Failed  : {failed_count}")
print(f"Skipped : {skip_count}")
print(f"Duplicate log : {duplicate_log}")
print(f"Failed log    : {failed_log}")

Total images: 38439
Workers: 32
[100/38439] success=100 failed=0 skip=0
[200/38439] success=200 failed=0 skip=0
[300/38439] success=300 failed=0 skip=0
[400/38439] success=400 failed=0 skip=0
[500/38439] success=500 failed=0 skip=0
[600/38439] success=600 failed=0 skip=0
[700/38439] success=700 failed=0 skip=0
[800/38439] success=800 failed=0 skip=0
[900/38439] success=900 failed=0 skip=0
[1000/38439] success=1000 failed=0 skip=0
[1100/38439] success=1100 failed=0 skip=0
[1200/38439] success=1200 failed=0 skip=0
[1300/38439] success=1300 failed=0 skip=0
[1400/38439] success=1400 failed=0 skip=0
[1500/38439] success=1500 failed=0 skip=0
[1600/38439] success=1600 failed=0 skip=0
[1700/38439] success=1700 failed=0 skip=0
[1800/38439] success=1800 failed=0 skip=0
[1900/38439] success=1900 failed=0 skip=0
[2000/38439] success=2000 failed=0 skip=0
[2100/38439] success=2100 failed=0 skip=0
[2200/38439] success=2200 failed=0 skip=0
[2300/38439] success=2300 failed=0 skip=0
[2400/38439] success

## Tải chung vào 1 folder (để nhìn mắt cho xôm, cái này k cần tải cũng đc), gồm cả duplicate (khác bệnh nhân,... trường hợp này không gặp khi đã chia folder)

In [3]:
import pandas as pd
import requests
import threading
import time

from pathlib import Path
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

# =====================================================
# CONFIG
# =====================================================

EXCEL_FILE = r"C:\Users\lebat\Documents\Github\xbone-net\data\CTCH\Data_CTCH.xlsx"

OUTPUT_DIR = Path(r"C:\Users\lebat\Documents\Github\xbone-net\data\CTCH\images_all")

MAX_WORKERS = 32
MAX_RETRIES = 8

OUTPUT_DIR.mkdir(exist_ok=True)

# =====================================================
# READ EXCEL
# =====================================================

image_df = pd.read_excel(
    EXCEL_FILE,
    sheet_name="Image Links"
)

image_df.columns = (
    image_df.columns
    .astype(str)
    .str.strip()
)

# =====================================================
# LOGS
# =====================================================

duplicate_log = OUTPUT_DIR / "duplicate_files_folderall.txt"
failed_log = OUTPUT_DIR / "failed_urls_folderall.txt"

duplicate_log.write_text("", encoding="utf-8")
failed_log.write_text("", encoding="utf-8")

filename_lock = Lock()
log_lock = Lock()

# =====================================================
# SESSION
# =====================================================

thread_local = threading.local()

def get_session():

    if not hasattr(thread_local, "session"):

        session = requests.Session()

        adapter = requests.adapters.HTTPAdapter(
            pool_connections=100,
            pool_maxsize=100
        )

        session.mount("http://", adapter)
        session.mount("https://", adapter)

        thread_local.session = session

    return thread_local.session

# =====================================================
# FILE NAME
# =====================================================

def get_filename(url):

    filename = unquote(
        Path(
            urlparse(url).path
        ).name
    )

    return filename

# =====================================================
# UNIQUE FILEPATH
# =====================================================

def get_unique_filepath(filepath):

    with filename_lock:

        if not filepath.exists():
            return filepath, False

        stem = filepath.stem
        suffix = filepath.suffix

        counter = 1

        while True:

            candidate = (
                filepath.parent /
                f"{stem}_{counter}{suffix}"
            )

            if not candidate.exists():
                return candidate, True

            counter += 1

# =====================================================
# DOWNLOAD
# =====================================================

def download_file(url, filepath):

    session = get_session()

    last_error = None

    for attempt in range(MAX_RETRIES):

        try:

            response = session.get(
                url,
                timeout=(10, 180),
                stream=True
            )

            response.raise_for_status()

            with open(filepath, "wb") as f:

                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        f.write(chunk)

            return True

        except Exception as e:

            last_error = e

            wait_time = min(
                2 ** attempt,
                60
            )

            time.sleep(wait_time)

    raise last_error

# =====================================================
# PROCESS
# =====================================================

def process_row(idx, row):

    try:

        url = row["Link tải ảnh"]

        if pd.isna(url):
            return "missing"

        filename = get_filename(url)

        if not filename:
            filename = f"image_{idx}.jpg"

        filepath = OUTPUT_DIR / filename

        filepath, duplicated = (
            get_unique_filepath(filepath)
        )

        if duplicated:

            with log_lock:

                with open(
                    duplicate_log,
                    "a",
                    encoding="utf-8"
                ) as f:

                    f.write(
                        f"{filename} -> {filepath.name}\n"
                    )

        # Resume:
        # Nếu file đã tồn tại và có dữ liệu
        if (
            filepath.exists()
            and filepath.stat().st_size > 0
        ):
            return "skip"

        download_file(
            url,
            filepath
        )

        return "success"

    except Exception:

        with log_lock:

            with open(
                failed_log,
                "a",
                encoding="utf-8"
            ) as f:

                f.write(
                    f"{idx}\t{url}\n"
                )

        return "failed"

# =====================================================
# RUN
# =====================================================

total = len(image_df)

success_count = 0
failed_count = 0
skip_count = 0

print(f"Total images: {total}")
print(f"Workers: {MAX_WORKERS}")

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = [

        executor.submit(
            process_row,
            idx,
            row
        )

        for idx, row in image_df.iterrows()
    ]

    completed = 0

    for future in as_completed(futures):

        completed += 1

        result = future.result()

        if result == "success":
            success_count += 1

        elif result == "failed":
            failed_count += 1

        elif result == "skip":
            skip_count += 1

        if completed % 100 == 0:

            print(
                f"[{completed}/{total}] "
                f"success={success_count} "
                f"failed={failed_count} "
                f"skip={skip_count}"
            )

print("\n==========================")
print("DOWNLOAD FINISHED")
print("==========================")
print(f"Total   : {total}")
print(f"Success : {success_count}")
print(f"Failed  : {failed_count}")
print(f"Skipped : {skip_count}")
print(f"Duplicate log : {duplicate_log}")
print(f"Failed log    : {failed_log}")

Total images: 38439
Workers: 32
[100/38439] success=100 failed=0 skip=0
[200/38439] success=200 failed=0 skip=0
[300/38439] success=300 failed=0 skip=0
[400/38439] success=400 failed=0 skip=0
[500/38439] success=500 failed=0 skip=0
[600/38439] success=600 failed=0 skip=0
[700/38439] success=700 failed=0 skip=0
[800/38439] success=800 failed=0 skip=0
[900/38439] success=900 failed=0 skip=0
[1000/38439] success=1000 failed=0 skip=0
[1100/38439] success=1100 failed=0 skip=0
[1200/38439] success=1200 failed=0 skip=0
[1300/38439] success=1300 failed=0 skip=0
[1400/38439] success=1400 failed=0 skip=0
[1500/38439] success=1500 failed=0 skip=0
[1600/38439] success=1600 failed=0 skip=0
[1700/38439] success=1700 failed=0 skip=0
[1800/38439] success=1800 failed=0 skip=0
[1900/38439] success=1900 failed=0 skip=0
[2000/38439] success=2000 failed=0 skip=0
[2100/38439] success=2100 failed=0 skip=0
[2200/38439] success=2200 failed=0 skip=0
[2300/38439] success=2300 failed=0 skip=0
[2400/38439] success